# Insurance Cost Prediction using Machine Learning

This project aims to predict individual medical insurance charges based on demographic and lifestyle factors such as age, gender, BMI, smoking status, number of children, and region.

The project follows a complete machine learning workflow including:

- Data Loading and Exploration
- Feature Engineering
- Data Preprocessing
- Model Training
- Model Evaluation
- Model Comparison

The objective is to identify the model that provides the most accurate prediction of insurance costs.

------------------
## Import libraries

In [83]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import (mean_absolute_error, root_mean_squared_error, r2_score)
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor

-----------------
## Load dataset

In [84]:
insur = pd.read_csv(r"C:\Users\chand\Desktop\AIML Learning\Insurance price prediction\insurance.csv")
insur.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


-----------------------------
## Exploratory Data Analysis (EDA)

Before building machine learning models, it is important to understand the dataset and identify patterns that may influence insurance charges.

The following analyses are performed:

- Dataset Shape and Structure
- Average Charges by Gender
- Average Charges by Region
- Impact of Smoking on Charges
- Effect of Number of Children on Charges

These insights help understand which features may contribute significantly to insurance cost prediction.

In [102]:
print("Shape:", insur.shape)
print("Info:",insur.info())
print(insur.describe())

Shape: (1338, 8)
<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   age        1338 non-null   int64  
 1   sex        1338 non-null   int64  
 2   bmi        1338 non-null   float64
 3   children   1338 non-null   int64  
 4   smoker     1338 non-null   int64  
 5   region     1338 non-null   int64  
 6   charges    1338 non-null   float64
 7   age_group  1338 non-null   int64  
dtypes: float64(2), int64(6)
memory usage: 83.8 KB
Info: None
               age          sex  ...       charges    age_group
count  1338.000000  1338.000000  ...   1338.000000  1338.000000
mean     39.207025     0.494768  ...  13270.422265     2.785501
std      14.049960     0.500160  ...  12110.011237     0.896865
min      18.000000     0.000000  ...   1121.873900     1.000000
25%      27.000000     0.000000  ...   4740.287150     2.000000
50%      39.000000     0.000000  ...   938

In [86]:
# Average charges by gender
insur.groupby("sex")["charges"].mean()

sex
female    12569.578844
male      13956.751178
Name: charges, dtype: float64

In [87]:
# Average charges by region
insur.groupby("region")["charges"].mean()

region
northeast    13406.384516
northwest    12417.575374
southeast    14735.411438
southwest    12346.937377
Name: charges, dtype: float64

In [88]:
# Average charges of smoker and non-smoker
insur.groupby(["smoker"])["charges"].mean()

smoker
no      8434.268298
yes    32050.231832
Name: charges, dtype: float64

In [89]:
# Average charges by no. of children
insur.groupby("children")["charges"].mean()

children
0    12365.975602
1    12731.171832
2    15073.563734
3    15355.318367
4    13850.656311
5     8786.035247
Name: charges, dtype: float64

--------------------------------------------------
## Feature Engineering

Feature engineering is the process of creating meaningful features that can improve model performance.

A new feature called **Age Group** is created by categorizing individuals into different age ranges:

- Child
- Young Adult
- Middle Aged
- Elder

This transformation helps the model capture age-related patterns more effectively.

In [90]:
insur["age_group"] = pd.cut(insur["age"], bins=(0,18,35,50,100),labels=["child","young","middle","elder"])
insur["age_group"]

0        young
1        child
2        young
3        young
4        young
         ...  
1333    middle
1334     child
1335     child
1336     young
1337     elder
Name: age_group, Length: 1338, dtype: category
Categories (4, str): ['child' < 'young' < 'middle' < 'elder']

----------------------------------
## Data Preprocessing

Machine learning models require numerical input data.

The categorical features:

- Sex
- Smoker
- Region
- Age Group

are converted into numerical representations using label encoding/mapping techniques.

This step ensures compatibility with machine learning algorithms.

In [91]:
# Converting categorical data

insur["sex"] = insur["sex"].map({"male" : 0 ,"female" : 1})
insur["smoker"] = insur["smoker"].map({"no" : 0 ,"yes" : 1})
insur["region"] = insur["region"].map({"northeast" : 1, "northwest" : 2,"southeast" : 3, "southwest" : 4})
insur["age_group"] = insur["age_group"].map({"child" : 1, "young" : 2, "middle" : 3, "elder" : 4})
insur["age_group"] =insur["age_group"].astype(int)
insur.head()

,age,sex,bmi,children,smoker,region,charges,age_group
0,19,1,27.900,0,1,4,16884.92400,2
1,18,0,33.770,1,0,3,1725.55230,1
2,28,0,33.000,3,0,3,4449.46200,2
3,33,0,22.705,0,0,2,21984.47061,2
4,32,0,28.880,0,0,2,3866.85520,2


----------------------------------
## Train-Test Split

The dataset is divided into training and testing subsets.

- Training Set: Used to train the machine learning models.
- Testing Set: Used to evaluate model performance on unseen data.

This helps assess how well the models generalize to new data.

In [92]:
X = insur[["age","sex","bmi","children","smoker","region","age_group"]]
y = insur["charges"]
train_X,test_X,train_y,test_y = train_test_split(X, y, test_size= 0.2, random_state= 42)
train_X.dtypes

age            int64
sex            int64
bmi          float64
children       int64
smoker         int64
region         int64
age_group      int64
dtype: object

-------------------------------------
# Model Selection and Comparison

Multiple regression algorithms are trained and evaluated to determine the best-performing model.

The following models are compared:

1. Linear Regression
2. Random Forest Regressor
3. Decision Tree Regressor
4. Gradient Boosting Regressor
5. XGBoost Regressor

Evaluation Metrics:

- MAE (Mean Absolute Error)
- RMSE (Root Mean Squared Error)
- R² Score

The model with the highest predictive performance is selected as the final model.

#### 1. Linear Regression

In [93]:
print("Linear Regression")
lr_model = LinearRegression()
lr_model.fit(train_X,train_y)
lr_pred = lr_model.predict(test_X)
lr_mae = mean_absolute_error(lr_pred,test_y)
print("Mean Absolute Error:" ,lr_mae)
lr_rmse = root_mean_squared_error(lr_pred,test_y)
print("Root Mean Squared Error:",lr_rmse)
lr_r2 = r2_score(test_y,lr_pred)
print("r2 score:",lr_r2)

Linear Regression
Mean Absolute Error: 4186.082922879223
Root Mean Squared Error: 5787.83983328686
r2 score: 0.784223100479522


#### 2. Random Forest Regressor

In [94]:
print("Random Forest Regressor")
rf_model = RandomForestRegressor(random_state = 42)
rf_model.fit(train_X,train_y)
rf_pred = rf_model.predict(test_X)
rf_mae = mean_absolute_error(test_y,rf_pred)
print("Mean absolute Error:",rf_mae)
rf_rmse = root_mean_squared_error(test_y,rf_pred)
print("Root Mean Squared Error:",rf_rmse)
rf_r2 = r2_score(test_y,rf_pred)
print("r2 score:",rf_r2)

Random Forest Regressor
Mean absolute Error: 2543.5792716389624
Root Mean Squared Error: 4616.816114822856
r2 score: 0.8627042499267218


#### 3. Decision Tree Regressor

In [95]:
print("Decision Tree Regressor")
dt_model = DecisionTreeRegressor(random_state= 42)
dt_model.fit(train_X,train_y)
dt_pred = dt_model.predict(test_X)
dt_mae = mean_absolute_error(test_y,dt_pred)
print("Mean absolute Error:",dt_mae)
dt_rmse = root_mean_squared_error(test_y,dt_pred)
print("Root Mean Squared Error:",dt_rmse)
dt_r2 = r2_score(test_y,dt_pred)
print("r2 score:",dt_r2)

Decision Tree Regressor
Mean absolute Error: 2996.5669129813436
Root Mean Squared Error: 6509.302345939772
r2 score: 0.7270765653044822


#### 4. Gradient Boosting Regressor

In [96]:
print("Gradient Boosting Regressor")
gb_model = GradientBoostingRegressor(random_state= 42)
gb_model.fit(train_X,train_y)
gb_pred = gb_model.predict(test_X)
gb_mae = mean_absolute_error(gb_pred,test_y)
print("Mean Absolute Error:",gb_mae)
gb_rmse = root_mean_squared_error(gb_pred,test_y)
print("Root Mean Squared Error:",gb_rmse)
gb_r2 = r2_score(gb_pred,test_y)
print("r2 score:",gb_r2)

Gradient Boosting Regressor
Mean Absolute Error: 2447.186430323401
Root Mean Squared Error: 4352.171442805892
r2 score: 0.8613423327095462


#### 5. XGBoost Model

In [97]:
print("XGBoost Model")
xgb = XGBRegressor(random_state=42)
xgb.fit(train_X,train_y)
xgb_pred = xgb.predict(test_X)
xgb_mae = mean_absolute_error(xgb_pred,test_y)
print("Mean Absolute Error:",xgb_mae)
xgb_rmse = root_mean_squared_error(xgb_pred,test_y)
print("Root Mean Squared Error:",xgb_rmse)
xgb_r2 = r2_score(xgb_pred,test_y)
print("r2 score:",xgb_r2)


XGBoost Model
Mean Absolute Error: 2791.8325179517183
Root Mean Squared Error: 4822.991168492682
r2 score: 0.8445770940219712


-------------------------------------------
## Model Performance Comparison

The table below summarizes the performance of all trained regression models.

Comparison is based on:

- Lower MAE → Better average prediction accuracy
- Lower RMSE → Better handling of large errors
- Higher R² Score → Better explanatory power

This comparison helps identify the most suitable model for insurance cost prediction.

In [101]:
dic = {"Linear Regression" : [lr_mae,lr_rmse,lr_r2],
         "Random Forest Regressor": [rf_mae,rf_rmse,rf_r2],
         "Decision Tree Regressor": [dt_mae,dt_rmse,dt_r2],
         "Gradient Boosting Regressor" : [gb_mae,gb_rmse,gb_r2],
         "XGB Regressor" : [xgb_mae,xgb_rmse,xgb_r2]}

comparision_table = pd.DataFrame(dic,index=["MEA","RMSE","R2"]).T
comparision_table.sort_values(by="R2",ascending= False)

,MEA,RMSE,R2
Random Forest Regressor,2543.579272,4616.816115,0.862704
Gradient Boosting Regressor,2447.186430,4352.171443,0.861342
XGB Regressor,2791.832518,4822.991168,0.844577
Linear Regression,4186.082923,5787.839833,0.784223
Decision Tree Regressor,2996.566913,6509.302346,0.727077


---------------------------------------------------
# Conclusion

This project focused on predicting insurance charges using multiple regression algorithms.

The workflow included:

- Data Exploration and Visualization
- Feature Engineering
- Data Preprocessing
- Model Training and Evaluation

Five regression models were trained and compared:

1. Linear Regression
2. Decision Tree Regressor
3. Random Forest Regressor
4. Gradient Boosting Regressor
5. XGBoost Regressor

Based on the evaluation metrics (MAE, RMSE, and R² Score), **XGBoost Regressor achieved the best overall performance**, producing the highest R² score and the lowest prediction error among all models.

This indicates that XGBoost was able to capture the complex relationships between customer characteristics and insurance charges more effectively than the other models.

Therefore, XGBoost Regressor was selected as the final model for insurance cost prediction.

----------------------------
# Future Improvements

Potential enhancements include:

- Hyperparameter tuning using GridSearchCV.
- Cross-validation for more robust evaluation.
- Additional feature engineering.
- Testing advanced boosting techniques such as LightGBM and CatBoost.
- Deploying the model as a web application using Flask or FastAPI.

These improvements may further increase prediction accuracy and real-world usability.